In [9]:
import os, gc, json, time, re, numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F, torchvision.transforms as T
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, roc_curve, auc, accuracy_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
from transformers import ASTConfig, ASTModel
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════
# 📦 CONFIGURACIÓN CENTRAL
# ═══════════════════════════════════════════════════════════════════
class Config:
    SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
    SPLITS_DIR   = Path("Ansiedad/fase3/splits")
    SINTETICOS   = Path("Ansiedad/fase3/sinteticos")
    RESULTS_ROOT = Path("Ansiedad/fase3/resultado_final_cv")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    VAL_FRAC = 0.20
    THR_GRID = np.arange(0.05, 0.96, 0.02)
    RANDOM_STATE = 42

    HP = {
        "CNN2D": {"batch": 128, "epochs": 30, "patience": 5, "lr": 1e-4},
        "AST":   {"batch": 32,  "epochs": 20, "patience": 4, "lr": 2e-5, "n_unfreeze": 3}
    }
    AST_PRETRAINED = "MIT/ast-finetuned-audioset-10-10-0.4593"

    def __init__(self, candidates):
        self.candidates = candidates
        self.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════
# 🗃️ DATASETS (Corregido para soportar retornos de Stem en Inferencia)
# ═══════════════════════════════════════════════════════════════════
CNN_TF_TRAIN = T.Compose([T.ColorJitter(0.1,0.1), T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
CNN_TF_VAL   = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
AST_TF       = T.Compose([T.Grayscale(num_output_channels=1), T.ToTensor(), T.Normalize([0.5],[0.5])])

class SpecDataset(Dataset):
    def __init__(self, stems, img_map, transform, mode="CNN", ast_h=None, target_w=None, synthetic_paths=None, return_stem=False):
        self.samples = []
        self.transform = transform
        self.mode = mode
        self.ast_h = ast_h
        self.target_w = target_w
        self.return_stem = return_stem

        for stem in stems:
            if stem not in img_map or not img_map[stem]: continue
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]: 
                # BUG 1 CORREGIDO: Guardamos el string del stem si estamos en modo inferencia
                if self.return_stem:
                    self.samples.append((p, stem))
                else:
                    self.samples.append((p, label))
        
        # Sintéticos solo entran en entrenamiento (donde return_stem siempre es False)
        if synthetic_paths and not self.return_stem:
            for p in synthetic_paths: 
                self.samples.append((p, 1))

    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = Image.open(path).convert("RGB")
        t = self.transform(img)
        
        if self.mode == "AST":
            # ✅ FIX: Manejo robusto de dimensión de canal
            # AST_TF produce (1, H, W) o a veces (H, W, 1)
            if t.dim() == 3:
                # Buscar y eliminar la dimensión de canal (size=1)
                if t.shape[0] == 1:
                    t = t[0]  # (1, H, W) -> (H, W)
                elif t.shape[1] == 1:
                    t = t.squeeze(1)  # (H, 1, W) -> (H, W)
                elif t.shape[2] == 1:
                    t = t.squeeze(2)  # (H, W, 1) -> (H, W)
                else:
                    # Fallback: si ninguna dim es 1, asumir que ya es (H, W, C) y tomar primer canal
                    t = t[0] if t.shape[0] <= t.shape[1] and t.shape[0] <= t.shape[2] else t[..., 0]
            
            # Verificación de seguridad
            if t.dim() != 2:
                raise ValueError(f"AST Dataset: expected 2D tensor (H, W), got {t.dim()}D with shape {t.shape}")
            
            # Padding de altura si es necesario
            if self.ast_h is not None and t.shape[0] < self.ast_h:
                t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
            # Padding de ancho si es necesario
            if self.target_w is not None and t.shape[1] < self.target_w:
                t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
            
            # AST espera formato (time, freq) = (W, H)
            return t.permute(1, 0), target
        
        return t, target

# ═══════════════════════════════════════════════════════════════════
# 🧠 MODELOS (CNN2D y AST)
# ═══════════════════════════════════════════════════════════════════
class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()
        def conv_block(ic, oc, n):
            layers = []
            for i in range(n):
                in_ch = ic if i == 0 else oc
                layers += [
                    nn.Conv2d(in_ch, oc, kernel_size=3, padding=1),
                    nn.BatchNorm2d(oc), 
                    nn.ReLU(inplace=True)
                ]
            layers += [nn.MaxPool2d(2), nn.Dropout2d(0.2)]
            return nn.Sequential(*layers)
        
        self.features = nn.Sequential(
            conv_block(3, 32, 2), 
            conv_block(32, 64, 2), 
            conv_block(64, 128, 3), 
            conv_block(128, 256, 3), 
            conv_block(256, 256, 3)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): 
        return self.classifier(self.pool(self.features(x))).squeeze(1)


class ASTClassifier(nn.Module):
    def __init__(self, ast_h, target_w, n_unfreeze=3):
        super().__init__()
        fp, tp = (ast_h - 16) // 10 + 1, (target_w - 16) // 10 + 1
        cfg = ASTConfig(
            num_mel_bins=ast_h, max_length=target_w,
            hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=3072,
            patch_size=16, frequency_stride=10, time_stride=10
        )
        self.ast = ASTModel(cfg)
        pretrained = ASTModel.from_pretrained(Config.AST_PRETRAINED)
        
        self.ast.encoder.load_state_dict(pretrained.encoder.state_dict(), strict=False)
        self.ast.layernorm.load_state_dict(pretrained.layernorm.state_dict(), strict=False)
        self.ast.embeddings.patch_embeddings.load_state_dict(
            pretrained.embeddings.patch_embeddings.state_dict(), strict=False)
        
        self.ast.embeddings.cls_token.data = pretrained.embeddings.cls_token.data.clone()
        self.ast.embeddings.distillation_token.data = pretrained.embeddings.distillation_token.data.clone()
        
        old_pos = pretrained.embeddings.position_embeddings.data
        sp, pt, h = old_pos[:, :2, :], old_pos[:, 2:, :], old_pos.shape[-1]
        pt = pt.reshape(1, 12, 101, h).permute(0, 3, 1, 2)
        pt = F.interpolate(pt.float(), size=(fp, tp), mode='bilinear', align_corners=False)
        pt = pt.permute(0, 2, 3, 1).reshape(1, fp * tp, h)
        self.ast.embeddings.position_embeddings = nn.Parameter(torch.cat([sp, pt], dim=1))
        
        del pretrained
        gc.collect()
        
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1), nn.Sigmoid()
        )
        for p in self.parameters(): p.requires_grad = False
        for layer in self.ast.encoder.layer[-n_unfreeze:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.ast.layernorm.parameters(): p.requires_grad = True
        for p in self.classifier.parameters(): p.requires_grad = True
        
    def forward(self, x): 
        return self.classifier(self.ast(input_values=x).last_hidden_state[:, 0, :]).squeeze(1)

# ═══════════════════════════════════════════════════════════════════
# 🔄 GENERADOR SINTÉTICO (VAE Corregido con Aislamiento de Folds)
# ═══════════════════════════════════════════════════════════════════
class VAE(nn.Module):
    def __init__(self, in_h, in_w, dim_z=128):
        super().__init__()
        self.in_h, self.in_w = ((in_h + 15) // 16) * 16, ((in_w + 15) // 16) * 16
        self.dim_z = dim_z
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)
        )
        flat_dim = 256 * (self.in_h // 16) * (self.in_w // 16)
        self.fc_mu = nn.Linear(flat_dim, dim_z)
        self.fc_lv = nn.Linear(flat_dim, dim_z)
        self.fc_dec = nn.Linear(dim_z, flat_dim)
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, x):
        if x.shape[2:] != (self.in_h, self.in_w):
            x = F.interpolate(x, size=(self.in_h, self.in_w), mode='bilinear', align_corners=False)
        h = self.enc_conv(x).flatten(1)
        mu, logvar = self.fc_mu(h), self.fc_lv(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
        recon = self.dec_conv(self.fc_dec(z).view(-1, 256, self.in_h // 16, self.in_w // 16))
        if recon.shape != x.shape:
            recon = F.interpolate(recon, size=x.shape[2:], mode='bilinear', align_corners=False)
        return recon, mu, logvar
    
    def sample(self, n, device):
        z = torch.randn(n, self.dim_z, device=device)
        with torch.no_grad():
            return self.dec_conv(self.fc_dec(z).view(-1, 256, self.in_h // 16, self.in_w // 16))


class SyntheticGenerator:
    @staticmethod
    def train_and_generate(train_c1_paths, filter_name, duration, cfg_dict, n_needed, fold_num):
        # BUG 3 CORREGIDO: Carpeta de salida única por fold para prevenir fuga entre conjuntos de test cruzados
        out_dir = Config.SINTETICOS / filter_name / f"{duration}s" / f"fold_{fold_num:02d}" / "1_sintetico"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        existing = list(out_dir.glob("*.png"))
        if len(existing) >= n_needed:
            return existing[:n_needed]
        
        widths = [Image.open(p).size[0] for p in train_c1_paths[:50]] if train_c1_paths else [63]
        vae_w = int(np.median(widths))
        
        tf = T.Compose([T.Grayscale(1), T.Resize((cfg_dict["img_h"], vae_w)), T.ToTensor()])

        class VAEDS(Dataset):
            def __init__(self, paths, transform):
                self.paths = paths
                self.transform = transform
            def __len__(self): return len(self.paths)
            def __getitem__(self, i): return self.transform(Image.open(self.paths[i]).convert("RGB"))

        if not train_c1_paths or n_needed <= 0:
            return []
            
        dl = DataLoader(VAEDS(train_c1_paths, tf), batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
        vae = VAE(cfg_dict["img_h"], vae_w).to(Config.DEVICE)
        opt = optim.Adam(vae.parameters(), lr=1e-3)

        print(f"   [VAE] Entrenando VAE para Fold {fold_num}...")
        for ep in range(200):
            vae.train()
            for xb in dl:
                xb = xb.to(Config.DEVICE, non_blocking=True)
                opt.zero_grad()
                rec, mu, lv = vae(xb)
                if rec.shape != xb.shape:
                    rec = F.interpolate(rec, size=xb.shape[2:], mode='bilinear', align_corners=False)
                bce = F.binary_cross_entropy(rec, xb, reduction='sum') / xb.size(0)
                kl = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / xb.size(0)
                (bce + kl).backward()
                torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
                opt.step()

        vae.eval()
        gen_paths = list(existing)
        idx = len(existing)
        n_to_gen = n_needed - len(existing)
        
        if n_to_gen > 0:
            batch = 256
            while idx < n_needed:
                n = min(batch, n_needed - idx)
                with torch.no_grad():
                    imgs = vae.sample(n, Config.DEVICE)
                    imgs = F.interpolate(imgs, size=(cfg_dict["img_h"], vae_w), mode='bilinear', align_corners=False)
                    imgs = imgs.cpu().squeeze(1).numpy()
                
                for im in imgs:
                    rgb = (plt.cm.inferno(im)[:, :, :3] * 255).astype(np.uint8)
                    fname = out_dir / f"sintetico_{idx:06d}.png"
                    Image.fromarray(rgb, "RGB").save(fname)
                    gen_paths.append(fname)
                    idx += 1
        
        del vae; gc.collect(); torch.cuda.empty_cache()
        return gen_paths

# ═══════════════════════════════════════════════════════════════════
# 🎓 ENTRENAMIENTO & EVALUACIÓN (Corregido flujo de Padding y Probs)
# ═══════════════════════════════════════════════════════════════════
class ModelTrainer:
    def __init__(self, model_type, global_cfg):
        self.model_type = model_type
        self.global_cfg = global_cfg
        self.hp = global_cfg.HP[model_type]

    def build_model(self, ast_h=None, target_w=None):
        if self.model_type == "CNN2D": return CNN2D().to(Config.DEVICE)
        return ASTClassifier(ast_h, target_w, self.hp["n_unfreeze"]).to(Config.DEVICE)

    @staticmethod
    def calibrate_threshold(probs_dict, stems, labels):
        # Filtrar solo stems que tienen predicción
        valid = [(probs_dict[s], l) for s, l in zip(stems, labels) if s in probs_dict]
        if not valid:
            return 0.5  # Fallback seguro
        arr = np.array([p for p, _ in valid])
        valid_labels = np.array([l for _, l in valid])
        best_thr, best_f1 = 0.5, -1.0
        for thr in Config.THR_GRID:
            sc = f1_score(valid_labels, (arr > thr).astype(int), average='macro', zero_division=0)
            if sc > best_f1:
                best_f1, best_thr = sc, float(thr)
        return best_thr

    @staticmethod
    def predict_stems(model, stems, img_map, batch_size, mode, ast_h=None, target_w=None):
        model.eval()
        tf = CNN_TF_VAL if mode == "CNN2D" else AST_TF

        # Filtrar stems válidos antes de crear dataset
        valid_stems = [s for s in stems if s in img_map and img_map[s]]
        if not valid_stems:
            return {s: 0.5 for s in stems}  # Fallback

        # BUG 1 CORREGIDO: Activamos return_stem=True para recolectar strings válidos de los pacientes
        ds = SpecDataset(valid_stems, img_map, tf, mode, ast_h, target_w, return_stem=True)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
        
        probs = {s: [] for s in stems}
        with torch.no_grad():
            for xb, snames in dl:
                out = model(xb.to(Config.DEVICE)).view(-1).cpu().numpy()
                for p, s in zip(out, snames):
                    probs[s].append(float(p))
        return {s: float(np.mean(v)) for s, v in probs.items() if len(v) > 0}

    def train_and_evaluate_fold(self, inner_train_stems, inner_val_stems, outer_test_stems, img_map, cfg_dict, synth_paths):
        # BUG 2 CORREGIDO: Calculamos target_w ANTES de construir el Dataset de entrenamiento
        target_w = None
        if cfg_dict["model"] == "AST":
            valid_widths = [Image.open(img_map[s][0]).size[0] for s in inner_train_stems if s in img_map and img_map[s]]
            target_w = max(valid_widths) if valid_widths else 63

        ds_train = SpecDataset(
            inner_train_stems, img_map, CNN_TF_TRAIN, 
            mode=cfg_dict["model"], ast_h=cfg_dict.get("ast_h"), 
            target_w=target_w, synthetic_paths=synth_paths
        )
        dl_train = DataLoader(ds_train, batch_size=self.hp["batch"], shuffle=True, num_workers=4, pin_memory=True)

        # Optimización: contamos directamente sobre las muestras del dataset armado
        c0 = sum(len(img_map.get(s, [])) for s, l in zip(inner_train_stems, 
            [int(img_map[s][0].parent.name) for s in inner_train_stems if s in img_map]) 
            if l == 0 and s in img_map)
        
        c1_real = sum(len(img_map.get(s, [])) for s, l in zip(inner_train_stems, 
               [int(img_map[s][0].parent.name) for s in inner_train_stems if s in img_map]) 
               if l == 1 and s in img_map)
        
        c1 = c1_real + len(synth_paths)
        
        if c0 == 0 or c1 == 0:
            cw = torch.tensor([1.0, 1.0], device=Config.DEVICE)
        else:
            total = c0 + c1
            cw = torch.tensor([total / (2 * c0), total / (2 * c1)], device=Config.DEVICE)

        model = self.build_model(cfg_dict.get("ast_h"), target_w)
        
        if self.model_type == "AST":
            opt = optim.AdamW(model.parameters(), lr=self.hp["lr"], weight_decay=0.01)
            sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.hp["epochs"], eta_min=1e-7)
        else:
            opt = optim.Adam(model.parameters(), lr=self.hp["lr"])
            sched = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=3, min_lr=1e-7)
        
        criterion = nn.BCELoss(reduction='none')

        best_loss, pat_cnt, best_state = float('inf'), 0, None
        for ep in range(self.hp["epochs"]):
            model.train()
            ep_loss = 0.0
            for xb, yb in dl_train:
                xb, yb = xb.to(Config.DEVICE, non_blocking=True), yb.to(Config.DEVICE, non_blocking=True).float()
                opt.zero_grad()
                loss = (criterion(model(xb).view(-1), yb) * cw[yb.long()]).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                ep_loss += loss.item()
            
            avg_loss = ep_loss / len(dl_train)
            sched.step() if self.model_type == "AST" else sched.step(avg_loss)

            if avg_loss < best_loss:
                best_loss, pat_cnt = avg_loss, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                pat_cnt += 1
            if pat_cnt >= self.hp["patience"]: break

        model.load_state_dict(best_state)

        # Calibrar umbral en validación interna (stems reales únicamente)
        val_probs = self.predict_stems(model, inner_val_stems, img_map, self.hp["batch"], cfg_dict["model"], cfg_dict.get("ast_h"), target_w)
        val_labels = [int(img_map[s][0].parent.name) for s in inner_val_stems if s in val_probs]
        thr = self.calibrate_threshold(val_probs, inner_val_stems, val_labels)

        # Evaluar en el conjunto de test externo (Holdout del Fold)
        test_probs = self.predict_stems(model, outer_test_stems, img_map, self.hp["batch"], cfg_dict["model"], cfg_dict.get("ast_h"), target_w)
        
        true = np.array([int(img_map[s][0].parent.name) for s in outer_test_stems if s in test_probs])
        pred = np.array([1 if test_probs[s] > thr else 0 for s in outer_test_stems if s in test_probs])
        prob = np.array([test_probs[s] for s in outer_test_stems if s in test_probs])

        if len(np.unique(true)) > 1:
            fpr, tpr, _ = roc_curve(true, prob)
            roc_auc = auc(fpr, tpr)
        else:
            fpr, tpr, roc_auc = np.array([0, 1]), np.array([0, 1]), 0.5

        metrics = {
            "accuracy": accuracy_score(true, pred),
            "precision": precision_score(true, pred, zero_division=0),
            "recall": recall_score(true, pred, zero_division=0),
            "f1": f1_score(true, pred, zero_division=0),
            "f1_macro": f1_score(true, pred, average='macro', zero_division=0),
            "auc": float(roc_auc),
            "threshold": float(thr)
        }
        
        del model, ds_train, dl_train, best_state; gc.collect(); torch.cuda.empty_cache()
        return {
            "metrics": metrics, "test_probs": test_probs, "true_labels": true.tolist(),
            "pred_labels": pred.tolist(), "prob_scores": prob.tolist(),
            "roc_fpr": fpr.tolist(), "roc_tpr": tpr.tolist(), "stems": [s for s in outer_test_stems if s in test_probs]
        }

# ═══════════════════════════════════════════════════════════════════
# 🌐 ORQUESTRADOR DE CROSS-VALIDATION
# ═══════════════════════════════════════════════════════════════════
class CrossValidationOrchestrator:
    def __init__(self, config):
        self.cfg = config

    def build_img_map(self, filter_name, duration):
        root = Config.SPECS_ROOT / filter_name / f"{duration}s"
        img_map = {}
        for c in ["0", "1"]:
            cls_dir = root / c
            if not cls_dir.exists(): continue
            for p in sorted(cls_dir.glob("*.png")):
                stem = re.sub(r'_seg\d+$', '', p.stem)
                img_map.setdefault(stem, []).append(p)
        return img_map

    @staticmethod
    def save_fold_visualizations(metrics_data, candidate_id, fold_num, output_dir, condition="Ansiedad"):
        fold_dir = output_dir / candidate_id / f"fold_{fold_num:02d}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        true, pred, fpr, tpr = map(np.array, [metrics_data["true_labels"], metrics_data["pred_labels"], metrics_data["roc_fpr"], metrics_data["roc_tpr"]])
        thr = metrics_data["metrics"]["threshold"]

        # Matriz de Confusión
        cm = confusion_matrix(true, pred)
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=[f"Sin {condition}", f"Con {condition}"], yticklabels=[f"Sin {condition}", f"Con {condition}"])
        ax.set_title(f"Confusion Matrix — Fold {fold_num}\nThreshold: {thr:.2f}")
        ax.set_ylabel("Real"); ax.set_xlabel("Predicción")
        plt.tight_layout(); plt.savefig(fold_dir / "confusion_matrix.png", dpi=150); plt.close()

        # Curva ROC
        roc_auc = auc(fpr, tpr) if len(np.unique(true)) > 1 else 0.0
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, lw=2.5, color='navy', label=f"AUC = {roc_auc:.3f}")
        ax.fill_between(fpr, tpr, alpha=0.08, color='navy')
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        ax.set_xlabel("Tasa de Falsos Positivos"); ax.set_ylabel("Tasa de Verdaderos Positivos")
        ax.set_title(f"ROC Curve — Fold {fold_num}\n{candidate_id}")
        ax.legend(loc='lower right'); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(fold_dir / "roc_curve.png", dpi=150); plt.close()

        with open(fold_dir / "fold_data.json", "w") as f:
            json.dump({"candidate_id": candidate_id, "fold_num": fold_num, "threshold": thr, "metrics": metrics_data["metrics"]}, f, indent=2)
        return fold_dir

    def run_fold(self, train_idx, test_idx, all_stems, all_labels, cand, fold_num):
        tr_stems, te_stems = np.array(all_stems)[train_idx], np.array(all_stems)[test_idx]
        tr_labels = np.array(all_labels)[train_idx]

        # Inner split: 80% train / 20% val de datos reales
        sss = StratifiedShuffleSplit(n_splits=1, test_size=Config.VAL_FRAC, random_state=42)
        inner_tr_idx, inner_val_idx = next(sss.split(tr_stems, tr_labels))
        inner_tr = tr_stems[inner_tr_idx].tolist()
        inner_val = tr_stems[inner_val_idx].tolist()

        img_map = self.build_img_map(cand["filter"], cand["dur"])
        
        # BUG 4 CORREGIDO: Contamos segmentos (imágenes PNG) reales totales para calcular n_synth de forma correcta
        n_segs_c0 = sum(len(img_map.get(s, [])) for s, l in zip(inner_tr, tr_labels[inner_tr_idx]) if l == 0)
        n_segs_c1 = sum(len(img_map.get(s, [])) for s, l in zip(inner_tr, tr_labels[inner_tr_idx]) if l == 1)
        n_synth = max(0, n_segs_c0 - n_segs_c1)

        c1_real_tr = [s for s, l in zip(inner_tr, tr_labels[inner_tr_idx]) if l == 1]
        c1_paths_tr = [p for s in c1_real_tr for p in img_map.get(s, [])]

        # Pasamos fold_num para garantizar el aislamiento de la carpeta sintética
        synth_paths = SyntheticGenerator.train_and_generate(c1_paths_tr, cand["filter"], cand["dur"], cand, n_synth, fold_num)

        trainer = ModelTrainer(cand["model"], self.cfg)
        fold_results = trainer.train_and_evaluate_fold(inner_tr, inner_val, te_stems.tolist(), img_map, cand, synth_paths)

        viz_dir = self.save_fold_visualizations(fold_results, cand["id"], fold_num, self.cfg.RESULTS_ROOT)
        print(f"   📊 Visualizaciones guardadas en: {viz_dir}")

        del trainer, img_map; gc.collect(); torch.cuda.empty_cache()
        return fold_results["metrics"]

    def run_full_cv(self, n_splits=5):
        print(f"🚀 Iniciando Cross-Validation Externo ({n_splits} folds)...")
        fold_results = {c["id"]: [] for c in self.cfg.candidates}

        for cand in self.cfg.candidates:
            img_map = self.build_img_map(cand["filter"], cand["dur"])
            all_stems = sorted(img_map.keys())
            all_labels = np.array([int(img_map[s][0].parent.name) for s in all_stems])

            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            for fold, (train_idx, test_idx) in enumerate(skf.split(all_stems, all_labels)):
                print(f" 📊 {cand['id']} | Fold {fold+1}/{n_splits}")
                metrics = self.run_fold(train_idx, test_idx, all_stems, all_labels, cand, fold_num=fold+1)
                fold_results[cand["id"]].append(metrics)
                print(f"      ✅ F1-Macro: {metrics['f1_macro']:.3f} | AUC: {metrics['auc']:.3f}")

        for cid, folds in fold_results.items():
            means = {k: np.mean([f[k] for f in folds]) for k in folds[0]}
            stds = {k: np.std([f[k] for f in folds]) for k in folds[0]}
            print(f"\n🏆 {cid} (Promedio ± Std):")
            print(f"   F1-Macro: {means['f1_macro']:.3f} ± {stds['f1_macro']:.3f}")
            print(f"   AUC:      {means['auc']:.3f} ± {stds['auc']:.3f}")
            
            with open(self.cfg.RESULTS_ROOT / f"{cid}_cv_summary.json", "w") as f:
                json.dump({"means": means, "stds": stds, "folds": folds}, f, indent=2)
        return fold_results

# ═══════════════════════════════════════════════════════════════════
# ▶️ EJECUCIÓN
# ═══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    CANDIDATOS = [
        {"id": "CNN2D_logmel_2s", "model": "CNN2D", "filter": "log-mel", "dur": 2, "img_h": 128},
        {"id": "AST_gammatone_2s", "model": "AST", "filter": "gammatone", "dur": 2, "img_h": 128, "ast_h": 128},
        {"id": "AST_gammatone_3s", "model": "AST", "filter": "gammatone", "dur": 3, "img_h": 128, "ast_h": 128},
        {"id": "AST_gammatone_4s", "model": "AST", "filter": "gammatone", "dur": 4, "img_h": 128, "ast_h": 128}
    ]
    cfg = Config(CANDIDATOS)
    orchestrator = CrossValidationOrchestrator(cfg)
    resultados_cv = orchestrator.run_full_cv(n_splits=5)

🚀 Iniciando Cross-Validation Externo (5 folds)...
 📊 CNN2D_logmel_2s | Fold 1/5
   [VAE] Entrenando VAE para Fold 1...
   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/CNN2D_logmel_2s/fold_01
      ✅ F1-Macro: 0.429 | AUC: 0.333
 📊 CNN2D_logmel_2s | Fold 2/5
   [VAE] Entrenando VAE para Fold 2...
   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/CNN2D_logmel_2s/fold_02
      ✅ F1-Macro: 0.667 | AUC: 0.604
 📊 CNN2D_logmel_2s | Fold 3/5
   [VAE] Entrenando VAE para Fold 3...
   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/CNN2D_logmel_2s/fold_03
      ✅ F1-Macro: 0.385 | AUC: 0.604
 📊 CNN2D_logmel_2s | Fold 4/5
   [VAE] Entrenando VAE para Fold 4...
   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/CNN2D_logmel_2s/fold_04
      ✅ F1-Macro: 0.600 | AUC: 0.708
 📊 CNN2D_logmel_2s | Fold 5/5
   [VAE] Entrenando VAE para Fold 5...
   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/CNN2D_logmel_2s/

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14829.55it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/AST_gammatone_2s/fold_01
      ✅ F1-Macro: 0.417 | AUC: 0.396
 📊 AST_gammatone_2s | Fold 2/5
   [VAE] Entrenando VAE para Fold 2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16721.09it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/AST_gammatone_2s/fold_02
      ✅ F1-Macro: 0.619 | AUC: 0.771
 📊 AST_gammatone_2s | Fold 3/5
   [VAE] Entrenando VAE para Fold 3...


...

   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/AST_gammatone_4s/fold_03
      ✅ F1-Macro: 0.590 | AUC: 0.667
 📊 AST_gammatone_4s | Fold 4/5
   [VAE] Entrenando VAE para Fold 4...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17164.67it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/AST_gammatone_4s/fold_04
      ✅ F1-Macro: 0.200 | AUC: 0.604
 📊 AST_gammatone_4s | Fold 5/5
   [VAE] Entrenando VAE para Fold 5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20164.44it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   📊 Visualizaciones guardadas en: Ansiedad/fase3/resultado_final_cv/AST_gammatone_4s/fold_05
      ✅ F1-Macro: 0.167 | AUC: 0.667

🏆 CNN2D_logmel_2s (Promedio ± Std):
   F1-Macro: 0.526 ± 0.105
   AUC:      0.550 ± 0.127

🏆 AST_gammatone_2s (Promedio ± Std):
   F1-Macro: 0.406 ± 0.115
   AUC:      0.551 ± 0.130

🏆 AST_gammatone_3s (Promedio ± Std):
   F1-Macro: 0.564 ± 0.083
   AUC:      0.564 ± 0.113

🏆 AST_gammatone_4s (Promedio ± Std):
   F1-Macro: 0.337 ± 0.154
   AUC:      0.537 ± 0.145
